In [1]:
from torchvision import datasets, transforms
from PIL import Image
import shutil
import os
import random
def generate_mnist_samples(number: int, max_samples: int = 100, test_fraction: float = 0.2, output_dir: str = "../../tests/generated_samples") -> None:
    """
    Generate and save MNIST samples for a specified number, split into train and test sets.

    Args:
        number (int): The MNIST digit to generate samples for (0-9).
        max_samples (int, optional): The maximum number of samples to generate. Defaults to 100.
        test_fraction (float, optional): Fraction of samples to use for test set. Defaults to 0.2.
        output_dir (str, optional): The base output directory. Defaults to "../../tests/generated_samples".

    Returns:
        None
    """
    # Set up the output directories
    digit_output_dir = os.path.join(output_dir, f"mnist_{number}")
    train_dir = os.path.join(digit_output_dir, "train")
    test_dir = os.path.join(digit_output_dir, "test")
    # Remove existing directories with files inside
    if os.path.exists(digit_output_dir):
        shutil.rmtree(digit_output_dir)
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)

    # Download and load MNIST dataset
    transform = transforms.Compose([transforms.ToTensor()])
    mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

    randomize = True

    # Filter for only the specified number
    filtered_dataset = [(img, label) for img, label in mnist_train if label == number]
    filtered_dataset = filtered_dataset[:max_samples] if not randomize else random.sample(filtered_dataset, min(max_samples, len(filtered_dataset)))
    
    # Calculate split indices
    test_size = int(len(filtered_dataset) * test_fraction)
    train_size = len(filtered_dataset) - test_size

    # Split dataset into train and test
    train_dataset = filtered_dataset[:train_size]
    test_dataset = filtered_dataset[train_size:]

    # Generate and save training images
    for i, (img, _) in enumerate(train_dataset):
        pil_img = transforms.ToPILImage()(img.squeeze())
        pil_img = pil_img.resize((100, 100), Image.BILINEAR)
        pil_img.save(os.path.join(train_dir, f"mnist_{number}_{i:05d}.png"))

    # Generate and save test images with separate counter
    for i, (img, _) in enumerate(test_dataset):
        pil_img = transforms.ToPILImage()(img.squeeze())
        pil_img = pil_img.resize((100, 100), Image.BILINEAR)
        pil_img.save(os.path.join(test_dir, f"mnist_{number}_{i:05d}.png"))

    print(f"Generated {train_size} training images and {test_size} test images of the number {number}")
    print(f"Training images in: {train_dir}")
    print(f"Test images in: {test_dir}")

In [2]:
def wait_for_kafka_idle(topic: str, idle_timeout: int = 30, bootstrap_servers: str = "localhost:29092") -> None:
    """
    Wait until a Kafka topic has been idle (no new messages) for the specified duration.
    
    Args:
        topic (str): Name of the Kafka topic to monitor
        idle_timeout (int, optional): Time in seconds to wait for no activity before considering idle. Defaults to 30.
        bootstrap_servers (str, optional): Kafka bootstrap servers. Defaults to "localhost:29092".
        
    Returns:
        None
    """
    from kafka import KafkaConsumer
    import time
    
    # Create consumer
    consumer = KafkaConsumer(
        topic,
        bootstrap_servers=bootstrap_servers,
        auto_offset_reset='latest',
        enable_auto_commit=True,
        group_id=None,
        consumer_timeout_ms=1000  # 1 second timeout for poll()
    )
    
    try:
        last_message_time = time.time()
        print(f"Monitoring topic {topic} for {idle_timeout} seconds of inactivity...")
        
        while True:
            # Try to get message
            messages = consumer.poll(timeout_ms=1000)
            current_time = time.time()
            
            if messages:
                # Reset timer if we got messages
                last_message_time = current_time
                print("Messages received, resetting idle timer...")
            else:
                # Check if we've been idle long enough
                idle_duration = current_time - last_message_time
                if idle_duration >= idle_timeout:
                    print(f"No messages received for {idle_timeout} seconds. Topic {topic} is idle.")
                    return
                
                if idle_duration >= 5:  # Only print every 5 seconds
                    print(f"No messages for {int(idle_duration)} seconds...")
    
    finally:
        consumer.close()



In [3]:
import subprocess

def train_mnist(class_number: int, subclass: int | None = None, samples: int | None = None, is_prepared_samples: bool = False) -> None:
    """
    Train MNIST classifier for a specific class and optional subclass.
    
    Args:
        class_number (int): The main class number
        subclass (int | None): Optional subclass number
        samples (int | None): Number of samples to use
        is_prepared_samples (bool): Whether to use prepared samples
    """
    if is_prepared_samples:
        if subclass is not None:
            subprocess.run(["make", f"train_prepared_samples_{class_number}", str(subclass)], cwd="../../")
        else:
            subprocess.run(["make", f"train_prepared_samples_{class_number}"], cwd="../../")
    else:
        generate_mnist_samples(class_number, subclass, max_samples=samples)
        subprocess.run(["make", f"train_mnist_{class_number}"], cwd="../../")
        
    wait_for_kafka_idle(topic="contour-analysis-output-topic", idle_timeout=10, bootstrap_servers="localhost:29092")
    subprocess.run(["make", "post_process", str(class_number) + (f"_{subclass}" if subclass is not None else ""), f"mnist-{class_number}"], cwd="../../")

In [4]:
from neo4j import GraphDatabase

def clean_neo4j_db() -> None:
    """Cleans all nodes and relationships from Neo4j database"""
    uri = "bolt://localhost:7687"
    user = "neo4j"
    password = "111122223333"

    driver = GraphDatabase.driver(uri, auth=(user, password))
    with driver.session() as session:
        # Delete all nodes and relationships
        session.run("MATCH (n) DETACH DELETE n")
    driver.close()


def clean_kafka_topics() -> None:
    """Deletes all messages from Kafka topics by recreating them"""
    from kafka.admin import KafkaAdminClient, NewTopic
    from kafka.errors import TopicAlreadyExistsError, UnknownTopicOrPartitionError
    
    topics = [
        "connector-output-topic",
        "line-detector-output-topic", 
        "angle-point-detector-output-topic",
        "skeletonization-output-topic",
        "contour-analysis-output-topic",
        "classification-output-topic",
        "dlq-topic"
    ]
    
    admin_client = KafkaAdminClient(bootstrap_servers="localhost:29092")
    
    # Delete existing topics
    for topic in topics:
        try:
            admin_client.delete_topics([topic])
            logging.info(f"Deleted topic: {topic}")
        except UnknownTopicOrPartitionError:
            logging.info(f"Topic {topic} does not exist")
    
    time.sleep(5)  # Wait for topics to be fully deleted
    
    # Recreate topics
    topic_list = []
    for topic in topics:
        topic_list.append(NewTopic(
            name=topic,
            num_partitions=1,
            replication_factor=1
        ))
    
    for topic in topic_list:
        try:
            admin_client.create_topics([topic])
            logging.info(f"Created topic: {topic.name}")
        except TopicAlreadyExistsError:
            logging.warning(f"Topic {topic.name} already exists")
    
    admin_client.close()


In [5]:
import os
import logging
from kafka import KafkaConsumer
import json
import time
from typing import Dict, Any, List, Tuple
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix,
    precision_recall_fscore_support,
    accuracy_score,
)
import pandas as pd
from datetime import datetime

# Constants
KAFKA_BOOTSTRAP_SERVERS = "localhost:29092"
KAFKA_TOPICS = [
    "connector-output-topic",
    "line-detector-output-topic",
    "angle-point-detector-output-topic",
    "skeletonization-output-topic",
    "contour-analysis-output-topic",
    "classification-output-topic",
    "dlq-topic",
]
KAFKA_CONSUMER_TIMEOUT_MS = 1000
KAFKA_POLL_TIMEOUT_MS = 1000
CLASSIFICATION_TIMEOUT_SECS = 30
CONFUSION_MATRIX_FIGSIZE = (10, 7)
TRAINING_RESULTS_DIR = "training_results"

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


def save_confusion_matrix(cm: np.ndarray, classes: List[str], run_dir: str) -> None:
    """Plot and save confusion matrix."""
    plt.figure(figsize=CONFUSION_MATRIX_FIGSIZE)
    plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    plt.title("Confusion Matrix")
    plt.colorbar()

    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    # Add text annotations
    thresh = cm.max() / 2.0
    for i, j in np.ndindex(cm.shape):
        plt.text(
            j,
            i,
            format(cm[i, j], "d"),
            horizontalalignment="center",
            color="white" if cm[i, j] > thresh else "black",
        )

    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()

    # Save plot in run directory
    plt.savefig(os.path.join(run_dir, "confusion_matrix.png"))
    plt.close()


def test_mnist_all(classes: List[int]) -> Tuple[Dict[str, Any], List[str], List[str]]:
    """Test MNIST classification for all classes and calculate overall metrics.

    Args:
        classes: List of class numbers to test

    Returns:
        Tuple containing results dict, true labels and predicted labels
    """
    all_results: Dict[str, Any] = {}
    all_y_true: List[str] = []
    all_y_pred: List[str] = []
    incorrect_results: List[Dict[str, Any]] = []  # Track incorrect classifications

    # Create run directory with timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = os.path.join(TRAINING_RESULTS_DIR, f"run_{timestamp}")
    os.makedirs(run_dir, exist_ok=True)

    for class_number in classes:
        test_folder = f"../../tests/generated_samples/mnist_{class_number}/test"
        test_images = [
            f for f in os.listdir(test_folder) if f.endswith((".png", ".jpg", ".jpeg"))
        ]
        nuclio_volume_path = (
            f"/opt/nuclio/shared_storage/generated_samples/mnist_{class_number}/test"
        )
        test_images = [os.path.join(nuclio_volume_path, f) for f in test_images]

        consumer = KafkaConsumer(
            "classification-output-topic",
            "dlq-topic",
            bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
            auto_offset_reset="latest",
            enable_auto_commit=True,
            value_deserializer=lambda x: json.loads(x.decode("utf-8")),
            group_id=f"test-mnist",
            consumer_timeout_ms=KAFKA_CONSUMER_TIMEOUT_MS,
        )

        for image_file in test_images:
            logger.debug(f"Processing image: {image_file}")
            os.system(f"cd ../../ && make classify {image_file}")

            start_time = time.time()
            dlq_result = False
            expected_name = f"mnist-{class_number}"

            while time.time() - start_time < CLASSIFICATION_TIMEOUT_SECS:
                try:
                    messages = consumer.poll(timeout_ms=KAFKA_POLL_TIMEOUT_MS)
                    for topic_partition, msgs in messages.items():
                        for msg in msgs:
                            if msg.topic == "dlq-topic":
                                dlq_result = True
                                logger.warning(
                                    f"Image {image_file} failed and went to DLQ"
                                )
                                incorrect_results.append(
                                    {
                                        "image_id": msg.value["value"]["parameters"][
                                            "image_id"
                                        ],
                                        "image_path": image_file,
                                        "expected": expected_name,
                                        "predicted": "error",
                                        "result": "DLQ",
                                    }
                                )
                                break
                            elif msg.topic == "classification-output-topic":
                                result = msg.value

                                # Verify the result corresponds to the current image
                                if (
                                    result["image_path"].split("/")[-1]
                                    != image_file.split("/")[-1]
                                ):
                                    logger.warning(
                                        f"Received result for different image. Expected {image_file}, got {result['parameters']['image_path']}"
                                    )
                                    continue

                                all_results[image_file] = result

                                if "classification_results" in result:
                                    top_result = sorted(
                                        result["classification_results"],
                                        key=lambda x: x["combined_score"],
                                        reverse=True,
                                    )[0]
                                    predicted_name = top_result["concept_name"]
                                    all_y_true.append(expected_name)
                                    all_y_pred.append(predicted_name)

                                    if predicted_name != expected_name:
                                        incorrect_results.append(
                                            {
                                                "image_id": result["image_id"],
                                                "image_path": image_file,
                                                "expected": expected_name,
                                                "predicted": predicted_name,
                                                "result": result[
                                                    "classification_results"
                                                ],
                                            }
                                        )

                                    logger.debug(
                                        f"Image {image_file}: {'Correct' if predicted_name == expected_name else 'Incorrect'} "
                                        f"classification - got {predicted_name}, expected {expected_name}"
                                    )
                                break
                    if dlq_result or image_file in all_results:
                        break
                except Exception as e:
                    logger.error(f"Error reading from Kafka: {e}")
                    time.sleep(0.1)
                    continue

            if time.time() - start_time >= CLASSIFICATION_TIMEOUT_SECS:
                logger.warning(f"Timeout waiting for classification of {image_file}")
                all_results[image_file] = "timeout"
                incorrect_results.append(
                    {
                        "image_id": os.path.basename(image_file),
                        "image_path": image_file,
                        "expected": expected_name,
                        "predicted": "timeout",
                        "result": "timeout",
                    }
                )

        consumer.close()

    # Calculate overall metrics
    total = len(test_images) * len(classes)
    failed_dlq = sum(1 for r in incorrect_results if r["result"] == "DLQ")
    successful = len(all_y_true)  # Only count actual classifications

    # Save incorrect results to CSV
    if incorrect_results:
        incorrect_df = pd.DataFrame(incorrect_results)
        incorrect_df.to_csv(os.path.join(run_dir, "incorrect_results.csv"), index=False)

    if successful > 0:
        # Calculate overall metrics excluding DLQ results
        labels = sorted(list(set(all_y_true + all_y_pred)))
        overall_precision, overall_recall, overall_f1, _ = (
            precision_recall_fscore_support(
                all_y_true, all_y_pred, labels=labels, average="weighted"
            )
        )
        overall_accuracy = accuracy_score(all_y_true, all_y_pred)

        # Calculate per-class metrics
        class_precision, class_recall, class_f1, support = (
            precision_recall_fscore_support(
                all_y_true, all_y_pred, labels=labels, average=None
            )
        )

        # Save overall metrics to CSV
        metrics_df = pd.DataFrame(
            {
                "Metric": [
                    "Total Images",
                    "Failed (DLQ)",
                    "Successfully Classified",
                    "Success Rate (%)",
                    "Accuracy (%)",
                    "Precision (%)",
                    "Recall (%)",
                    "F1 Score (%)",
                ],
                "Value": [
                    total,
                    failed_dlq,
                    successful,
                    (successful / (total - failed_dlq)) * 100,
                    overall_accuracy * 100,
                    overall_precision * 100,
                    overall_recall * 100,
                    overall_f1 * 100,
                ],
            }
        )
        metrics_df.to_csv(os.path.join(run_dir, "metrics.csv"), index=False)

        # Save per-class metrics to CSV
        per_class_data = []
        for i, label in enumerate(labels):
            if label not in ["error", "timeout"]:  # Skip non-class labels
                per_class_data.append(
                    {
                        "Class": label,
                        "Precision (%)": class_precision[i] * 100,
                        "Recall (%)": class_recall[i] * 100,
                        "F1 Score (%)": class_f1[i] * 100,
                        "Support": support[i],
                    }
                )

        per_class_df = pd.DataFrame(per_class_data)
        per_class_df.to_csv(os.path.join(run_dir, "per_class_metrics.csv"), index=False)

        logger.info("\nOverall Classification Metrics:")
        logger.info(f"Total images across all classes: {total}")
        logger.info(f"Failed (DLQ): {failed_dlq}")
        logger.info(f"Successfully classified: {successful}")
        logger.info(f"Overall success rate: {(successful/(total-failed_dlq))*100:.2f}%")
        logger.info(f"Overall accuracy: {overall_accuracy*100:.2f}%")
        logger.info(f"Overall precision: {overall_precision*100:.2f}%")
        logger.info(f"Overall recall: {overall_recall*100:.2f}%")
        logger.info(f"Overall F1 Score: {overall_f1*100:.2f}%")

        logger.info("\nPer-Class Metrics:")
        for i, label in enumerate(labels):
            if label not in ["error", "timeout"]:
                logger.info(f"\n{label}:")
                logger.info(f"Precision: {class_precision[i]*100:.2f}%")
                logger.info(f"Recall: {class_recall[i]*100:.2f}%")
                logger.info(f"F1 Score: {class_f1[i]*100:.2f}%")
                logger.info(f"Support: {support[i]}")

        # Generate and save confusion matrix
        cm = confusion_matrix(all_y_true, all_y_pred, labels=labels)
        save_confusion_matrix(cm, labels, run_dir)
    else:
        logger.warning("No successful classifications to calculate metrics")

    return all_results, all_y_true, all_y_pred

In [ ]:
train_set_sizes = [20, 30, 40, 50, 100, 200, 500]
classes = [1, 2, 3, 4, 5, 6, 7]

for size in train_set_sizes:
    clean_neo4j_db()
    clean_kafka_topics()
    for class_number in classes:
        train_mnist(class_number, size)
    test_mnist_all(classes)

In [11]:
classes_to_subclasses = {
    1: [1, 2, 3],
    2: [1, 2],
    3: [1],
    # 4: [1, 2],
    # 5: [1],
    # 6: [1],
    # 7: [1],
    # 8: [1],
    # 9: [1],
}

clean_kafka_topics()
clean_neo4j_db()

for class_number, subclasses in classes_to_subclasses.items():
    generate_mnist_samples(class_number, max_samples=10, test_fraction=1)
    for subclass in subclasses:
        train_mnist(class_number=class_number, subclass=subclass, samples=None, is_prepared_samples=True)

test_mnist_all(classes_to_subclasses.keys())

INFO:kafka.conn:<BrokerConnection node_id=bootstrap-0 host=localhost:29092 <connecting> [IPv6 ('::1', 29092, 0, 0)]>: connecting to localhost:29092 [('::1', 29092, 0, 0) IPv6]
INFO:kafka.conn:Probing node bootstrap-0 broker version
INFO:kafka.conn:<BrokerConnection node_id=bootstrap-0 host=localhost:29092 <connecting> [IPv6 ('::1', 29092, 0, 0)]>: Connection complete.
INFO:kafka.conn:Broker version identified as 2.5.0
INFO:kafka.conn:Set configuration api_version=(2, 5, 0) to skip auto check_version requests on startup
INFO:kafka.conn:Probing node bootstrap-0 broker version
INFO:kafka.conn:Broker version identified as 2.5.0
INFO:kafka.conn:Set configuration api_version=(2, 5, 0) to skip auto check_version requests on startup
INFO:kafka.conn:<BrokerConnection node_id=1001 host=192.168.0.105:29092 <connecting> [IPv4 ('192.168.0.105', 29092)]>: connecting to 192.168.0.105:29092 [('192.168.0.105', 29092) IPv4]
INFO:kafka.conn:Probing node 1001 broker version
INFO:kafka.conn:<BrokerConnecti

Generated 0 training images and 10 test images of the number 1
Training images in: ../../tests/generated_samples/mnist_1/train
Test images in: ../../tests/generated_samples/mnist_1/test
Running training script for prepared samples class 1 subclass 1... 
Sending data to connector... 
Images processed and sent to Kafka
Data sent to connector successfully. 
Training script completed. 


INFO:kafka.conn:Broker version identified as 2.5.0
INFO:kafka.conn:Set configuration api_version=(2, 5, 0) to skip auto check_version requests on startup
INFO:kafka.consumer.subscription_state:Updating subscribed topics to: ('contour-analysis-output-topic',)
INFO:kafka.consumer.subscription_state:Updated partition assignment: [TopicPartition(topic='contour-analysis-output-topic', partition=0)]
INFO:kafka.conn:<BrokerConnection node_id=1001 host=192.168.0.105:29092 <connecting> [IPv4 ('192.168.0.105', 29092)]>: connecting to 192.168.0.105:29092 [('192.168.0.105', 29092) IPv4]
INFO:kafka.conn:<BrokerConnection node_id=1001 host=192.168.0.105:29092 <connecting> [IPv4 ('192.168.0.105', 29092)]>: Connection complete.
INFO:kafka.conn:<BrokerConnection node_id=bootstrap-0 host=localhost:29092 <connected> [IPv6 ('::1', 29092, 0, 0)]>: Closing connection. 


Monitoring topic contour-analysis-output-topic for 10 seconds of inactivity...
Messages received, resetting idle timer...
Messages received, resetting idle timer...
Messages received, resetting idle timer...
Messages received, resetting idle timer...
Messages received, resetting idle timer...
Messages received, resetting idle timer...
Messages received, resetting idle timer...
Messages received, resetting idle timer...
Messages received, resetting idle timer...
Messages received, resetting idle timer...
Messages received, resetting idle timer...
Messages received, resetting idle timer...
Messages received, resetting idle timer...
Messages received, resetting idle timer...
Messages received, resetting idle timer...
Messages received, resetting idle timer...
Messages received, resetting idle timer...
Messages received, resetting idle timer...
Messages received, resetting idle timer...
Messages received, resetting idle timer...
Messages received, resetting idle timer...
Messages received,

INFO:kafka.conn:<BrokerConnection node_id=1001 host=192.168.0.105:29092 <connected> [IPv4 ('192.168.0.105', 29092)]>: Closing connection. 
ERROR:kafka.consumer.fetcher:Fetch to node 1001 failed: Cancelled: <BrokerConnection node_id=1001 host=192.168.0.105:29092 <connected> [IPv4 ('192.168.0.105', 29092)]>


...

Classifying image: /opt/nuclio/shared_storage/generated_samples/mnist_3/test/mnist_3_00002.png 
Image sent for classificationClassification request sent to connector. 


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   163  100    29  100   134   9183  42431 --:--:-- --:--:-- --:--:-- 54333


Classifying image: /opt/nuclio/shared_storage/generated_samples/mnist_3/test/mnist_3_00003.png 
Image sent for classificationClassification request sent to connector. 


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   163  100    29  100   134   6842  31618 --:--:-- --:--:-- --:--:-- 40750
INFO:kafka.coordinator:Stopping heartbeat thread
INFO:kafka.coordinator:Leaving consumer group (test-mnist).
INFO:kafka.conn:<BrokerConnection node_id=coordinator-1001 host=192.168.0.105:29092 <connected> [IPv4 ('192.168.0.105', 29092)]>: Closing connection. 
INFO:kafka.conn:<BrokerConnection node_id=1001 host=192.168.0.105:29092 <connected> [IPv4 ('192.168.0.105', 29092)]>: Closing connection. 
INFO:__main__:
Overall Classification Metrics:
INFO:__main__:Total images across all classes: 30
INFO:__main__:Failed (DLQ): 2
INFO:__main__:Successfully classified: 27
INFO:__main__:Overall success rate: 96.43%
INFO:__main__:Overall accuracy: 59.26%
INFO:__main__:Overall precision: 59.67%
INFO:__main__:Overall recall: 59.26%
INFO:__main__:Overall F1 Score: 57.5

({'/opt/nuclio/shared_storage/generated_samples/mnist_1/test/mnist_1_00006.png': 'timeout',
  '/opt/nuclio/shared_storage/generated_samples/mnist_1/test/mnist_1_00007.png': {'classification_results': [{'concept_id': '4dda2cfd06cffa460e95281324efd9f1d3df30d22054a8e60c1237eef1d04c2f',
     'concept_name': 'mnist-1',
     'raw_structural_score': 1.0,
     'raw_feature_score': 0.6316626866729586,
     'session_id': '1_3',
     'combined_score': 0.2883566363924879},
    {'concept_id': '5fd875d5fa9370aea683be3bfeb094fceb52acce9c8cee79392dbc3789851af4',
     'concept_name': 'mnist-1',
     'raw_structural_score': 0.4375,
     'raw_feature_score': 0.1140498671259616,
     'session_id': '1_1',
     'combined_score': 0.16878578105001604},
    {'concept_id': '0b38e263a92889765099df8617954366f2fae67fbed57912e59492242f428c4c',
     'concept_name': 'mnist-1',
     'raw_structural_score': 0.0,
     'raw_feature_score': 0.06445928609200093,
     'session_id': '1_2',
     'combined_score': 0.1375346650